# Sod Shock Tube (1D) -- resume

Resumes a run exported by `sod_1d.ipynb`/`sod_1d.py` from the last frame of
its `trajectory.h5` and keeps appending new frames to that *same* growing
file (`warpSPH.io.loadTrajectory`/`loadTrajectoryFrame` read it back;
`writeFrame` keeps writing to it) -- there is one export per run, resume
included, not a parallel set of files.

As in `sod_1d.ipynb`, the step loop below is unrolled rather than hidden
behind `warpSPH.runner.run()`, so it stays a place to hook in new logic, and
plotting calls `plotSod`/`plotSod_` directly rather than
`sodCase.setupPlot`/`updatePlot` -- see `sod_1d.ipynb`'s intro cell for why.


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sod import sodCase, states
from warpSPH.caseUtils import plotSod, plotSod_
from warpSPH.io import importConfigs, latestExportPath, loadTrajectory, loadTrajectoryFrame, writeFrame
from warpSPH.runner import CaseSpec, RunContext, encodeFrames

import os
import torch
import h5py
from tqdm.autonotebook import tqdm
from warpSPHIntegrators.integration import getIntegrator


In [ ]:
# Resuming reads its parameters back from the export, not from CaseSpec
# defaults -- edit these if you want to point at a different run or change
# how far/what it stores.
exportPath = None                # None -> newest run of the case
plot = True
store = True
plotInterval = 10
exportInterval = None            # None -> reuse the interval the run was exported with
t_limit = 0.3                    # simulated-time target for this resume

if exportPath is None:
    exportPath = latestExportPath(sodCase.defaults['caseName'])
print(f'resuming from {exportPath}')


In [ ]:
# Load the trajectory export: the static (masses/kinds/materials/UIDs) and
# per-frame (positions/velocities/densities/internalEnergies/supports)
# fields, plus the scheme bundle needed to keep stepping.
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

trajectoryFile, meta = loadTrajectory(exportPath, device, extraFields=sodCase.extraFields)
config, schemeConfig = importConfigs(os.path.join(exportPath, 'config.json'), meta['bundle'].importFunction)

# Sod's own params (gamma, left/right states, ...) were written once as
# top-level attrs by `writeInitialData`'s `extraData` -- read them back so
# `sodCase`'s hooks (which all read `ctx.param(...)`) see the run that
# actually ran, not the case's bare defaults.
params = {k: trajectoryFile.attrs[k] for k in sodCase.params if k in trajectoryFile.attrs}
spec = CaseSpec(caseName=sodCase.name, scheme=sodCase.scheme, params=params)

lastFrameIndex = int(meta['frameKeys'][-1].split('_')[1])
system, t = loadTrajectoryFrame(trajectoryFile, meta, len(meta['frameKeys']) - 1, schemeConfig=schemeConfig)
if exportInterval is None:
    exportInterval = float(trajectoryFile.attrs['exportInterval'])
trajectoryFile.close()

print(f'resuming from frame {lastFrameIndex}, t={t:.5f}')


In [ ]:
ctx = RunContext(
    spec=spec, case=sodCase, config=config, integrator=getIntegrator(config.integrationScheme),
    schemeConfig=schemeConfig, scheme=meta['scheme'], device=device, dtype=config.dtype, bundle=meta['bundle'],
    exportPath=exportPath,
)

runningState = system.initializeNewState()
dt = config.dt if isinstance(config.dt, float) else config.dt.cpu().item()
delta_t = t_limit - runningState.t
nSteps = int(delta_t / dt)
print(f"Running {nSteps} steps to reach t={t_limit:.5f}.")

# Direct plotSod/plotSod_ + plt.subplots(), the same way the pre-Case
# notebooks (e.g. 02-linear-wave.ipynb) always did -- sodCase.setupPlot's
# openWindow/pumpEvents indirection was confirmed (in sod_1d.ipynb) to break
# live updates in this environment even though the underlying plotSod/
# plotSod_ calls are identical, so this notebook skips that path entirely.
fig = axis = None
if plot:
    ctx.imagePath = os.path.join(exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    left, right = states(ctx)
    fig, axis = plotSod(runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                        ctx.param('gamma'), left, right,
                        plotReference=True, plotLabels=False, scatter=False, t_=runningState.t)

outFile = None
groups = None
if store:
    outFile = h5py.File(os.path.join(exportPath, 'trajectory.h5'), 'a')
    groups = (outFile['positions'], outFile['velocities'], outFile['densities'], outFile['times'],
             outFile['rigidBodyTrajectories']) + tuple(outFile[name] for name in sodCase.extraFields)
    storeSteps = max(1, int(exportInterval / dt))


In [ ]:
# The step loop, visible and editable -- same shape as sod_1d.ipynb's.
startIndex = lastFrameIndex + 1
for i in (tq := tqdm(range(startIndex, startIndex + nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sodCase.diagnostics(ctx, runningState)
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % plotInterval == 0 or i == startIndex + nSteps - 1):
        for ax in axis.flatten():
            ax.clear()
        plotSod_(fig, axis, runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                ctx.param('gamma'), left, right,
                plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == startIndex + nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=sodCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if plot:
    encodeFrames(ctx.imagePath, exportPath)
